In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 42
OUTPUTS = Path('outputs')
OUTPUTS.mkdir(exist_ok=True)
df = pd.read_csv('data/tips.csv')
print('Dataset shape:', df.shape)
display(df.head())

Dataset shape: (244, 7)


,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


## 1. Data source and feature meanings
The CSV is the real `tips` sample dataset bundled with Plotly Express, not synthetic data. Source: https://github.com/plotly/datasets/blob/master/tips.csv . It describes historical restaurant observations and is not representative of every restaurant.

| Column | Meaning | Role |
|---|---|---|
| total_bill | Bill in dollars, including tax | Numeric feature |
| tip | Recorded tip in dollars | Target |
| sex | Gender label recorded for payer | Categorical feature |
| smoker | Smoking status recorded | Categorical feature |
| day | Day of week | Categorical feature |
| time | Lunch or dinner | Categorical feature |
| size | Party size | Numeric feature |

The exercise uses all available input columns. Gender and smoking labels are included for assignment completeness; this does not establish that they cause different tips or justify using them in a deployed system.

In [2]:
print('Column types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
display(df.describe(include='all'))
required = ['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size']
assert list(df.columns) == required
# Exact repeated records cannot be distinguished from genuine identical bills.
# Remove them conservatively before splitting, to avoid overlap across partitions.
df = df.drop_duplicates().dropna(subset=['tip']).reset_index(drop=True)
assert (df[['total_bill', 'tip']] >= 0).all().all()
assert (df['size'] >= 1).all()
print('Rows after cleaning:', len(df))

Column types:
total_bill    float64
tip           float64
sex            object
smoker         object
day            object
time           object
size            int64
dtype: object

Missing values:
total_bill    0
tip           0
sex           0
smoker        0
day           0
time          0
size          0
dtype: int64

Duplicate rows: 1


,total_bill,tip,sex,smoker,day,time,size
count,244.000000,244.000000,244,244,244,244,244.000000
unique,NaN,NaN,2,2,4,2,NaN
top,NaN,NaN,Male,No,Sat,Dinner,NaN
freq,NaN,NaN,157,151,87,176,NaN
mean,19.785943,2.998279,NaN,NaN,NaN,NaN,2.569672
std,8.902412,1.383638,NaN,NaN,NaN,NaN,0.951100
min,3.070000,1.000000,NaN,NaN,NaN,NaN,1.000000
25%,13.347500,2.000000,NaN,NaN,NaN,NaN,2.000000
50%,17.795000,2.900000,NaN,NaN,NaN,NaN,2.000000
75%,24.127500,3.562500,NaN,NaN,NaN,NaN,3.000000


Rows after cleaning: 243


## 2. Visual exploration
Scatter plots show associations with the target; the pie chart describes the sample composition. Association is not proof of causation. Interactive HTML charts are saved alongside the notebook.

In [3]:
figures = {
    'bill_vs_tip': px.scatter(df, x='total_bill', y='tip', color='time',
                              hover_data=['day', 'size'], title='Bill amount and tip'),
    'party_size_vs_tip': px.scatter(df, x='size', y='tip', color='day',
                                   title='Party size and tip'),
    'bills_by_day': px.pie(df.groupby('day').size().reset_index(name='count'),
                          names='day', values='count', title='Bills by day'),
}
for name, figure in figures.items():
    figure.update_layout(template='plotly_white')
    figure.write_html(OUTPUTS / f'{name}.html', include_plotlyjs=True)
    display(figure)
print('Correlation between bill and tip:', round(df['total_bill'].corr(df['tip']), 3))

Correlation between bill and tip: 0.675


## 3. Hold out test data and build the pipeline
Use an 80/20 reproducible random split. This dataset has no dates, so the split estimates performance on similar observations; it is not a future-time forecast. The target is excluded from the inputs. Median/mode imputation and one-hot encoding are fitted within the pipeline, only on training data.

Linear regression is the specified model and a useful interpretable first approach to a continuous dollar target. Drop the first category to avoid redundant dummy columns; unseen categories are ignored when predicting. No feature scaling is needed for unregularized linear regression.

In [4]:
numeric = ['total_bill', 'size']
categorical = ['sex', 'smoker', 'day', 'time']
features = numeric + categorical
X = df[features]
y = df['tip']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)
numeric_steps = Pipeline([('imputer', SimpleImputer(strategy='median'))])
categorical_steps = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_steps, numeric),
    ('categorical', categorical_steps, categorical),
])
model = Pipeline([('preprocessor', preprocessor), ('regression', LinearRegression())])
print('Training rows:', len(X_train), '| Test rows:', len(X_test))
assert set(X_train.index).isdisjoint(X_test.index)

Training rows: 194 | Test rows: 49


## 4. Validate on the training partition
Five-fold cross-validation checks how variable performance is without using the held-out test set. Each fold refits preprocessing. A mean-tip baseline gives context: the model should improve over predicting the same tip for every bill. No tuning decisions are made on the test results.

In [5]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
validation = cross_validate(model, X_train, y_train, cv=cv,
    scoring={'r2': 'r2', 'mae': 'neg_mean_absolute_error'})
cv_results = pd.DataFrame({'fold': range(1, 6),
    'r2': validation['test_r2'], 'mae_dollars': -validation['test_mae']})
display(cv_results)
cv_results.to_csv(OUTPUTS / 'cross_validation.csv', index=False)
print('Mean CV R²:', cv_results.r2.mean())
print('CV R² standard deviation:', cv_results.r2.std(ddof=0))
print('Mean CV MAE ($):', cv_results.mae_dollars.mean())
model.fit(X_train, y_train)
baseline = DummyRegressor(strategy='mean').fit(X_train, y_train)

,fold,r2,mae_dollars
0,1,0.326283,0.651256
1,2,0.333495,0.802880
2,3,0.380751,0.742023
3,4,0.492284,0.583880
4,5,0.275116,1.068656


Mean CV R²: 0.3615857632523599
CV R² standard deviation: 0.0734324059161383
Mean CV MAE ($): 0.7697392192817659


## 5. Evaluate once on the held-out test set
`model.score()` returns **R²**, not classification accuracy. R² = 1 is perfect; 0 matches predicting the test-target mean; negative values are possible. MAE is the average absolute prediction error in dollars. RMSE gives more weight to larger errors.

In [6]:
predictions = model.predict(X_test)
baseline_predictions = baseline.predict(X_test)
metrics = {
    'train_r2': float(model.score(X_train, y_train)),
    'test_r2': float(model.score(X_test, y_test)),
    'test_mae_dollars': float(mean_absolute_error(y_test, predictions)),
    'test_rmse_dollars': float(np.sqrt(mean_squared_error(y_test, predictions))),
    'baseline_test_r2': float(baseline.score(X_test, y_test)),
    'baseline_test_mae_dollars': float(mean_absolute_error(y_test, baseline_predictions)),
    'cv_mean_r2': float(cv_results.r2.mean()),
    'cv_std_r2': float(cv_results.r2.std(ddof=0)),
    'cv_mean_mae_dollars': float(cv_results.mae_dollars.mean()),
    'train_rows': len(X_train), 'test_rows': len(X_test),
}
print(json.dumps(metrics, indent=2))
results = X_test.copy()
results['actual_tip'] = y_test
results['predicted_tip'] = predictions
results['residual'] = y_test - predictions
results.to_csv(OUTPUTS / 'test_predictions.csv', index_label='row_index')
assert np.isfinite(predictions).all()
assert len(predictions) == len(y_test)
fig = px.scatter(results, x='actual_tip', y='predicted_tip',
                 title='Actual versus predicted tips')
limits = [float(min(y_test.min(), predictions.min())), float(max(y_test.max(), predictions.max()))]
fig.add_shape(type='line', x0=limits[0], y0=limits[0], x1=limits[1], y1=limits[1])
fig.write_html(OUTPUTS / 'actual_vs_predicted.html', include_plotlyjs=True)
display(fig)
residual_fig = px.scatter(results, x='predicted_tip', y='residual', title='Residuals')
residual_fig.add_hline(y=0)
residual_fig.write_html(OUTPUTS / 'residuals.html', include_plotlyjs=True)
display(residual_fig)

{
  "train_r2": 0.44524209546033433,
  "test_r2": 0.5185656337107039,
  "test_mae_dollars": 0.7665074194589535,
  "test_rmse_dollars": 1.0830525484338127,
  "baseline_test_r2": -0.010736047217692724,
  "baseline_test_mae_dollars": 1.148265306122449,
  "cv_mean_r2": 0.3615857632523599,
  "cv_std_r2": 0.0734324059161383,
  "cv_mean_mae_dollars": 0.7697392192817659,
  "train_rows": 194,
  "test_rows": 49
}


## 6. Predict a new example and save the result
No fixed example was supplied, so use a $30 bill for four people at dinner on Sunday, with the recorded payer label Female and non-smoker. This is an estimated historical tip, not a recommended tip percentage.

In [7]:
example = pd.DataFrame([{
    'total_bill': 30.0, 'size': 4, 'sex': 'Female',
    'smoker': 'No', 'day': 'Sun', 'time': 'Dinner',
}])[features]
example_tip = float(model.predict(example)[0])
display(example)
print(f'Predicted tip: ${example_tip:.2f}')
print(f'Predicted percentage of bill: {100 * example_tip / 30:.1f}%')
saved = {'features': example.iloc[0].to_dict(), 'predicted_tip_dollars': example_tip}
(OUTPUTS / 'example_prediction.json').write_text(json.dumps(saved, indent=2))
(OUTPUTS / 'metrics.json').write_text(json.dumps(metrics, indent=2))
coefficient_names = model.named_steps['preprocessor'].get_feature_names_out()
coefficients = pd.DataFrame({'feature': coefficient_names,
    'coefficient': model.named_steps['regression'].coef_})
display(coefficients)
coefficients.to_csv(OUTPUTS / 'coefficients.csv', index=False)

,total_bill,size,sex,smoker,day,time
0,30.0,4,Female,No,Sun,Dinner


Predicted tip: $4.23
Predicted percentage of bill: 14.1%


,feature,coefficient
0,numeric__total_bill,0.085483
1,numeric__size,0.210989
2,categorical__sex_Male,0.018162
3,categorical__smoker_Yes,-0.052130
4,categorical__day_Sat,0.103283
5,categorical__day_Sun,0.167782
6,categorical__day_Thur,-0.049832
7,categorical__time_Lunch,0.194669
